In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
BRONZE_SCHEMA = "bronze"
RAW_PATH = f"/Volumes/{CATALOG}/default/raw_data"

DATASETS = ["flights", "weather", "airports"]

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}"
)

DataFrame[]

In [0]:
for dataset in DATASETS:
    path = f"{RAW_PATH}/{dataset}"

    files = dbutils.fs.ls(path)

    print(f"{dataset.upper()}")
    print(f"Path: {path}")
    print(f"Top-level entries: {len(files)}")
    print("-" * 40)

FLIGHTS
Path: /Volumes/workspace/default/raw_data/flights
Top-level entries: 3
----------------------------------------
WEATHER
Path: /Volumes/workspace/default/raw_data/weather
Top-level entries: 3
----------------------------------------
AIRPORTS
Path: /Volumes/workspace/default/raw_data/airports
Top-level entries: 1
----------------------------------------


In [0]:
def read_raw_parquet(dataset: str):
    path = f"{RAW_PATH}/{dataset}"

    if dataset == "airports":
        # Single Parquet file
        return spark.read.parquet(path)

    if dataset == "flights":
        # flights/year=2025/month=01/*.parquet
        return (
            spark.read
            .option("basePath", path)
            .parquet(path)
        )

    if dataset == "weather":
        # weather/year=2025/airport=ATL/*.parquet
        df = (
            spark.read
            .option("recursiveFileLookup", "true")
            .option("pathGlobFilter", "*.parquet")
            .parquet(path)
        )

        file_path = F.col("_metadata.file_path")

        for column, dtype in [
            ("year", "int"),
            ("airport", "string")
        ]:
            if column not in df.columns:
                value = F.regexp_extract(
                    file_path,
                    rf"/{column}=([^/]+)(?:/|$)",
                    1
                )

                df = df.withColumn(
                    column,
                    F.when(value != "", value.cast(dtype))
                )

        return df

    raise ValueError(f"Unknown dataset: {dataset}")

In [0]:
def ingest_to_bronze(dataset: str):
    target = f"{CATALOG}.{BRONZE_SCHEMA}.{dataset}"

    df = (
        read_raw_parquet(dataset)
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_dataset", F.lit(dataset))
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target)
    )

    print(f"Refreshed {target}")

In [0]:
for dataset in DATASETS:
    ingest_to_bronze(dataset)

Refreshed workspace.bronze.flights
Refreshed workspace.bronze.weather
Refreshed workspace.bronze.airports
